<a href="https://colab.research.google.com/github/natchanant-arch/Project_Savings_Cooperative/blob/First/Part4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ส่วนที่ 4 — จำลอง "ลูกค้า 1 คนเดินเข้าร้าน" แบบ step-by-step

In [ ]:
import random
import time

def simulate_customer_visit(txn_id, customer_name, pause=0.5):
    """จำลองขั้นตอนลูกค้า 1 คนเดินเข้าธนาคาร"""
    print("=" * 60)
    queue_no = f"A-{txn_id:03d}"
    print(f"🎫 [ผู้ออกบัตรคิว] คุณ '{customer_name}' กดรับบัตรคิว ได้หมายเลข: {queue_no}")

    # 1. สร้าง Member
    member = Member(member_id=100 + txn_id, customer_name=customer_name)

    # 2. 🎲 สุ่มยอดเงินตั้งต้น (100 - 3,000 บาท)
    initial_balance = round(random.uniform(100, 3000), 2)

    # 3. 🎲 สุ่มเลขบัญชีลูกค้า (รูปแบบ 123-4-56789-0)
    acc_p1 = random.randint(100, 999)
    acc_p2 = random.randint(1, 9)
    acc_p3 = random.randint(10000, 99999)
    random_account_no = f"{acc_p1}-{acc_p2}-{acc_p3:05d}-0"

    # 4. สร้าง Account
    account = Account(account_number=random_account_no, balance=initial_balance, owner=member)

    # 5. สุ่มประเภทรายการ และจำนวนเงิน
    service = random.choice(["ฝากเงิน", "ถอนเงิน", "โอนเงิน"])
    amount = random_amount()
    target_acc = f"987-6-{random.randint(10000, 99999)}-0" if service == "โอนเงิน" else None

    # 6. ประมวลผลธุรกรรม
    print(f"🔔 [เชิญหมายเลข {queue_no}] เข้าเคาน์เตอร์บริการ -> แจ้งทำรายการ: '{service}'")
    print("⚙️ เจ้าหน้าที่บันทึกข้อมูลเข้าระบบ (สถานะ: กำลังดำเนินการ)")

    if service == "ฝากเงิน":
        account.deposit(amount)
    elif service == "ถอนเงิน":
        account.withdraw(amount)
    elif service == "โอนเงิน":
        dummy_mem = Member(0, "ปลายทาง")
        dummy_acc = Account("987-6-00000-0", balance=0.0, owner=dummy_mem)
        account.transfer(dummy_acc, amount)

    # คิดดอกเบี้ย
    account.apply_interest()

    # 7. สร้าง Transaction
    txn = Transaction(
        txn_id=txn_id,
        account=account,
        transaction_type=service,
        amount=amount,
        target_account=target_acc
    )

    # แสดงรายละเอียดคำนวณ
    print("🖥️ เจ้าหน้าที่ตรวจสอบยอดเงินและประเภทรายการ:")
    is_success = explain_transaction_calculation(txn)

    # ปรับรูปแบบชื่อกรณีที่เป็น Tuple/List
    display_name = " ".join(customer_name) if isinstance(customer_name, (tuple, list)) else customer_name

    # Check เงื่อนไขพิมพ์สลิป (ลบ print ข้อความไม่สำเร็จใน else ออกเพื่อไม่ให้ซ้ำ)
    if is_success:
        print("✅ ทำรายการสำเร็จ!")
        print(f"🧾 สลิปบันทึกรายการ #{txn.txn_id}: คิว {txn.queue_number} | วันที่ {txn.txn_date} | เวลา {txn.time} | คุณ {display_name} | {service} | ยอด {format_currency(amount)} | ยอดคงเหลือสุทธิ {format_currency(account.balance)}")
    else:
        # ไม่ต้องใส่ print("❌ ทำรายการไม่สำเร็จ!") ซ้ำตรงนี้แล้ว
        print(f"🧾 สลิปบันทึกรายการ #{txn.txn_id}: คิว {txn.queue_number} | วันที่ {txn.txn_date} | เวลา {txn.time} | คุณ {display_name} | [รายการยกเลิก - ยอดเงินไม่พอ]")

    return txn

In [ ]:
# --- [ทดสอบเรียกใช้งานจริงกับลูกค้า 1 คน] ---
transaction_a = simulate_customer_visit(txn_id=1, customer_name="สมหญิง สายทอง", pause=0.5)